In [49]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# Topic 1 {-}

## 1) {-}

### (a) {-}

In [50]:
X = np.array([[-2, -1, 0, 1],
               [0, 1, -1, 1],
               [-1.98, 1.01, -2.00, 2.98]]).T
mu = np.mean(X, axis=0)
print("The sample mean vector is \nμ =\n", mu[:, np.newaxis])

The sample mean vector is 
μ =
 [[-0.5   ]
 [ 0.25  ]
 [ 0.0025]]


### (b) {-}

In [59]:
Xbar = X - mu
print("The centered data matrix is equal to,\n", Xbar)

The centered data matrix is equal to,
 [[-1.5    -0.25   -1.9825]
 [-0.5     0.75    1.0075]
 [ 0.5    -1.25   -2.0025]
 [ 1.5     0.75    2.9775]]


## 2) {-}

In [66]:
S = np.cov(Xbar.T)
print("The data covariance matrix is \nS =\n", S)

The data covariance matrix is 
S =
 [[1.66666667 0.16666667 1.97833333]
 [0.16666667 0.91666667 1.99583333]
 [1.97833333 1.99583333 5.94029167]]


## 3) {-}

In [ ]:
vals, vecs = np.linalg.eig(S)

print("The eigenvalues are found to be\n", vals, "\n")
print("Corresponding to the eigenvectors\n", vecs)

The eigenvalues are found to be
 [7.29734325e+00 1.22628082e+00 9.31245192e-07] 

Corresponding to the eigenvectors
 [[-0.32486427 -0.85504119 -0.4041878 ]
 [-0.29005702  0.49684878 -0.81792922]
 [-0.90018339  0.14847847  0.40941912]]


In [68]:
U = vecs[:, :2]
print(  "Since the eigenvalues are already sorted in ascending order, " \
        "the first two eigenvectors corresponds to the data projection matrix." \
        "The two principal components of the dataset are then \nU =\n", U)

Since the eigenvalues are already sorted in ascending order, the first two eigenvectors corresponds to the data projection matrix.The two principal components of the dataset are then 
U =
 [[-0.32486427 -0.85504119]
 [-0.29005702  0.49684878]
 [-0.90018339  0.14847847]]


## 4) {-}

In [87]:
M = 2
y = np.zeros((0, M))

for n, xn in enumerate(Xbar):
    yn = U.T @ xn
    print(f"The projection of x_{n+1} is\n{yn[:,np.newaxis]}\n")
    y = np.vstack((y, yn))

print("The final projection of all centered data points is then\n", y.T)

The projection of x_1 is
[[2.34442422]
 [0.86399102]]

The projection of x_2 is
[[-0.96204539]
 [ 0.94974924]]

The projection of x_3 is
[[ 2.00275638]
 [-1.34590971]]

The projection of x_4 is
[[-3.38513521]
 [-0.46783055]]

The final projection of all centered data points is then
 [[ 2.34442422 -0.96204539  2.00275638 -3.38513521]
 [ 0.86399102  0.94974924 -1.34590971 -0.46783055]]


## 5) {-}

Information is lost, but the loss is minimized by picking the eigenvectors corresponding to the largest eigen values, which represents the largest variances. This means that the projection includes the most relevant and correlated data. The distortion of the data can be calculated with the eigenvalues with
$$
\mathcal{J} = \sum_{i=M+1}^D \lambda_i
$$

In [90]:
print("The distortion is J =", sum(vals[M:]))

The distortion is J = 9.312451920341208e-07


# Topic 2 {-}

## 1) {-}

### (a) {-}

In [110]:
X = np.array([  [-1, -1, -1, 1, 1, 1],
                [-1, 1, 0, -1, 1, 0]])

PHI = np.vstack((np.ones(len(X.T)), X))

print("The design matrix for the dataset is \nΦ =\n", PHI)

The design matrix for the dataset is 
Φ =
 [[ 1.  1.  1.  1.  1.  1.]
 [-1. -1. -1.  1.  1.  1.]
 [-1.  1.  0. -1.  1.  0.]]


### (b) {-}

In [120]:
def sigmoid(a):
    return 1 / (1 + np.exp(-a))

def log_likelihood(t, y):
    likelihood = 0
    for tn, yn in zip(t, y):
        likelihood += tn * np.log(yn) + (1 - tn) * np.log(1 - yn)
    return likelihood.item()

w = np.array([[0.01, 0, 0]]).T
t = np.array([[0, 0, 0, 1, 1, 1]]).T
y = sigmoid(w.T @ PHI).T

print("The log-likelihood for this dataset is", log_likelihood(t, y))

The log-likelihood for this dataset is -4.158958083047174


In [121]:
# slide 344
def cross_entropy_error(t, y):
    E = 0
    for tn, yn in zip(t, y):
        E -= t[n] * np.log(yn) + (1 - tn) * np.log(1 - yn)
    return E.item()

print("The cross entropy error for this dataset is", cross_entropy_error(t, y))

The cross entropy error for this dataset is 6.223437124570761


## 2) {-}

In [173]:
N = len(X.T)

dEdw = 0
for n in range(1, N):
    dEdw += (sigmoid(w.T @ PHI[:, n-1:n]) - t[n]) * PHI[:, n-1:n]
    
print("The gradient of the error function with respect to w is \n∂E/∂w =\n", dEdw)

The gradient of the error function with respect to w is 
∂E/∂w =
 [[-0.4875001 ]
 [-1.50249998]
 [ 0.        ]]


## 3) {-}

In [175]:
H = 0
for n in range(1, N):
    H += PHI[:, n-1:n] @ PHI[:, n-1:n].T
print("The hessian matrix is \nH =\n", H)

The hessian matrix is 
H =
 [[ 5. -1.  0.]
 [-1.  5.  0.]
 [ 0.  0.  4.]]


## 4) {-}

### (a) {-}

In [176]:
w1 = w - np.linalg.pinv(H) @ dEdw
print("After one Newton-Raphson update the weight vector become \nw =\n", w1)

After one Newton-Raphson update the weight vector become 
w =
 [[0.17416669]
 [0.33333333]
 [0.        ]]


## 5) {-}

### (a) {-}

In [186]:
PHI[:,3:N]

array([[ 1.,  1.,  1.],
       [ 1.,  1.,  1.],
       [-1.,  1.,  0.]])

In [ ]:
print("The decision boundary for P(C1|x) =", PC1x)

The decision boundary for P(C1|x) = [[0.62422024 0.62422024 0.62422024]]


### (b) {-}

The test point belongs to class 0 since the point is under the value of PC1x.

# Topic 3 {-}

## 1) {-}

In [91]:
X = np.array([  [0, 1, 0, 1],
                [0, 0, 1, 1]])
t = np.array([[0, 1, 1, 2]]).T

def gram_matrix(x1, x2, l):
    N = len(x1)
    M = len(x2)
    matrix = np.zeros((N,M))
    for n in range(N):
        for m in range(M):
            matrix[n,m] = np.exp(- np.linalg.norm(x1[n] - x2[m])**2 / (2*l**2))
            
    return matrix

K = gram_matrix(X, X, l=1)
print("The gram matrix is \nK =\n", K)

The gram matrix is 
K =
 [[1.         0.36787944]
 [0.36787944 1.        ]]


## 2) {-}

## 3) {-}

## 4) {-}

# Topic 4 {-}

## 1) {-}

The neuron transformations and thereby forward propagation is calculated using the following equations:
$$
a = w^{T}x + w_{0}
$$
$$
z = h(a)
$$

In [ ]:
W_1 = np.array([[0.1, -0.5, 0.4],
                [0.8, -0.2, 0.1],
                [-0.4, 0.6, -0.3],
                [0.2, -0.1, 0.5]])
W_2 = np.array([[-0.3],
                [0.5],
                [-0.2],
                [0.7]])
weigths = [W_1, W_2]

x = np.array([[0.5, -0.2, 0.8]]).T

# slide 450
def neuron_transformation(x, W, h):
    xbias = np.vstack(([1.0], x))
    a = W.T @ xbias
    print("a =\n", a)
    z = h(a)
    print("z =\n", z)
    return a, z

def identity(a):
    return a

def forward(x, weigths, h, h_output=None):
    activations = []
    outputs = []
    z = x
    num_layers = len(weigths)

    for i, W in enumerate(weigths):
        if i == num_layers-1:
            h = h_output or h
            print(f"-- Layer {i+1} (output) calculation steps --")
        else:
            print(f"-- Layer {i+1} (hidden) calculation steps --")
        a, z = neuron_transformation(z, W, h)
        activations.append(a)
        outputs.append(z)
        print()
    return outputs[-1], activations, outputs

y, activations, outputs = forward(x, weigths, sigmoid, identity)
print("y =", y.item())

-- Layer 1 (hidden) calculation steps --
a =
 [[ 0.74]
 [-0.8 ]
 [ 0.91]]
z =
 [[0.67699586]
 [0.31002552]
 [0.71300016]]

-- Layer 2 (output) calculation steps --
a =
 [[0.47559294]]
z =
 [[0.47559294]]

y = 0.47559293827138116


## 2) {-}

In [16]:
def sum_of_squares_error(t, y):
    return 1/2 * np.sum(y - t)**2

t = np.array([[1.0]])
print("The sum-of-squared error for t=1.0 is E(t, y) =", sum_of_squares_error(t, y))

The sum-of-squared error for t=1.0 is E(t, y) = 0.1375013831954217


## 3) {-}

In [20]:
# slide 452
def sigmoid_derivative(a):
    z = sigmoid(a)
    return z * (1 - z)

def compute_all_gradients(x, t, weights, activations, outputs, y, h_derivative):
    num_layers = len(weights)
    gradients = [None] * num_layers
    deltas = [None] * num_layers

    deltas[-1] = y - t
    print(f"δ^({num_layers}) =\n", deltas[-1], "\n")
    

    for k in range(num_layers - 2, -1, -1):
        W_no_bias = weights[k + 1][1:, :]
        deltas[k] = (W_no_bias @ deltas[k + 1]) * h_derivative(activations[k])
        print(f"δ^({k+1}) =\n", deltas[k], "\n")
    for k in range(num_layers):
        if k == 0:
            z_prev = np.vstack(([1.0], x))
        else:
            z_prev = np.vstack(([1.0], outputs[k - 1]))

        gradients[k] = z_prev @ deltas[k].T
        print(f"∂E/∂W^({k+1}) =\n", gradients[k], "\n")
    return gradients, deltas

gradients, _ = compute_all_gradients(x, t, weigths, activations, outputs, y, sigmoid_derivative)

print(f"∂E/∂w^(2)_3,1 = {gradients[1][3, 0]}")
print(f"∂E/∂w^(1)_0,1 = {gradients[0][0, 0]}")

δ^(2) =
 [[-0.52440706]] 

δ^(1) =
 [[-0.05733669]
 [ 0.02243515]
 [-0.07511693]] 

∂E/∂W^(1) =
 [[-0.05733669  0.02243515 -0.07511693]
 [-0.02866835  0.01121758 -0.03755847]
 [ 0.01146734 -0.00448703  0.01502339]
 [-0.04586935  0.01794812 -0.06009355]] 

∂E/∂W^(2) =
 [[-0.52440706]
 [-0.35502141]
 [-0.16257957]
 [-0.37390232]] 

∂E/∂w^(2)_3,1 = -0.373902320360951
∂E/∂w^(1)_0,1 = -0.05733669291727434


## 4) {-}

The architecture provided contains a weight matrix ($w^{(1)} \in \mathbb{R}^{3 \times 4}$) with 9 weights and 3 biases, as well as one ($w^{(2)} \in \mathbb{R}^{4 \times 1}$) with 3 weights and 1 bias. In total that makes 16 weights and biases.

A network produces a sum-of-squares error equal to exactly 0 has overfitted the data, since there should always be some margin of error due to noise and other uncertainties.

If there are more parameters than datapoints, then the model is bound to overfit to the training data.